# RKNN-LLM Model Export (Google Colab)
Llama 3.2 1B Instruct를 8가지 양자화 설정으로 변환합니다.
변환된 .rkllm 파일을 Google Drive에 저장 후 RK3588으로 전송하세요.

In [ ]:
# 1. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.5 HuggingFace 로그인 (Llama 3.2는 gated model)
from huggingface_hub import login
login()  # 토큰 입력 프롬프트가 뜹니다

In [ ]:
# 2. rkllm-toolkit 설치
# rkllm-toolkit whl 파일을 Google Drive에 미리 업로드하세요
!pip install /content/drive/MyDrive/rkllm_toolkit-1.2.1b1-cp310-cp310-linux_x86_64.whl
!pip install transformers huggingface_hub

In [ ]:
# 3. 모델 다운로드
from huggingface_hub import snapshot_download
model_path = snapshot_download('meta-llama/Llama-3.2-1B-Instruct', local_dir='./Llama-3.2-1B-Instruct')
print(f'Downloaded to: {model_path}')

In [ ]:
# 4. 캘리브레이션 데이터 생성
import json
from datasets import load_dataset

dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
texts = [t for t in dataset['text'] if len(t.strip()) > 50]

samples = []
for text in texts[:100]:
    text = text.strip()
    mid = len(text) // 2
    samples.append({'input': text[:mid], 'target': text[mid:]})

with open('./data_quant.json', 'w') as f:
    json.dump(samples, f, ensure_ascii=False, indent=2)
print(f'Generated {len(samples)} calibration samples')

In [ ]:
# 7. R1 성공 확인 후 나머지 R2~R8 변환
for cid in ['R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']:
    try:
        export_one(cid)
    except Exception as e:
        print(f'[FAIL] {cid}: {e}')

# 결과 확인
import glob
files = glob.glob('/content/drive/MyDrive/rkllm_models/*.rkllm')
for f in sorted(files):
    print(f'{os.path.basename(f)}: {os.path.getsize(f)/(1024*1024):.1f} MB')

## 변환 완료 후
Google Drive의 `rkllm_models/` 폴더에 8개 `.rkllm` 파일이 생성됩니다.

RK3588으로 전송:
```bash
# Google Drive에서 로컬 PC로 다운로드 후:
scp rkllm_models/*.rkllm hyunho.son@<RK3588_IP>:/home/hyunho.son/Desktop/project/GPU-NPU-Benchmark/llm_quant_bench/rkllm_models/

# 또는 RK3588에서 직접:
mkdir -p ~/Desktop/project/GPU-NPU-Benchmark/llm_quant_bench/rkllm_models
```

전송 후 벤치마크:
```bash
cd ~/Desktop/project/GPU-NPU-Benchmark/llm_quant_bench
python benchmark/bench_rkllm.py --model_path ./rkllm_models/Llama-3.2-1B-Instruct_W8A8_RK3588.rkllm
python eval/ppl_rkllm.py --model_path ./rkllm_models/Llama-3.2-1B-Instruct_W8A8_RK3588.rkllm --tokenizer_path ./models/Llama-3.2-1B-Instruct
```

In [ ]:
# 6. 전체 변환 실행 (순차적)
for cid in CONFIGS:
    try:
        export_one(cid)
    except Exception as e:
        print(f'[FAIL] {cid}: {e}')

# 결과 확인
import glob
files = glob.glob('/content/drive/MyDrive/rkllm_models/*.rkllm')
for f in sorted(files):
    print(f'{os.path.basename(f)}: {os.path.getsize(f)/(1024*1024):.1f} MB')

## 변환 완료 후
Google Drive의 `rkllm_models/` 폴더에 8개 `.rkllm` 파일이 생성됩니다.

RK3588으로 전송:
```bash
scp -r /path/to/rkllm_models/ user@rk3588:/path/to/llm_quant_bench/rkllm_models/
```